# 01 · Raw data tour

A walk through the post-filter input tables in `data/00_raw/`, checking the counts reported in the Methods (*Data sources*, *Disease set*) and the Supplementary input-source tables. Every `assert` compares a recomputed value with the number printed in the manuscript.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "00_raw").is_dir())
D = REPO / "data"
RAW = D / "00_raw"
def pairs(a, b):
    return {tuple(sorted(x)) for x in zip(a, b)}

## Clinical-trial triples
One row per trial–disease–drug link of the clinical-trial disease–drug network (input to IxIDN, notebook 02).

In [2]:
t = pd.read_parquet(RAW / "trials/trial_triples.parquet")
print(t.shape)
print(t[["trial_key", "disease_id", "drug_id"]].nunique())
print({int(k): int(v) for k, v in t.max_phase.value_counts().sort_index().items()})
assert t.disease_id.nunique() == 1593

(1692970, 8)
trial_key     419240
disease_id      1593
drug_id        38389
dtype: int64
{1: 331441, 2: 639855, 3: 255753, 4: 426932}


## Diseases
`diseases.parquet` lists every disease ID used by any raw table, with membership flags. The MechDist network holds 13,021 diseases.

In [3]:
d = pd.read_parquet(RAW / "diseases/diseases.parquet")
print(d[["is_disease_indication", "in_dd_network", "in_mechdist_network", "in_mechdist_universe"]].sum().to_dict())
net = d[d.in_mechdist_network]
print(net.disease_id.str.extract(r"^([A-Z]+)")[0].replace({"DIS": "DIS_B (blinded)"}).value_counts().to_dict())
assert len(net) == 13021

{'is_disease_indication': 1555, 'in_dd_network': 1593, 'in_mechdist_network': 13021, 'in_mechdist_universe': 13147}
{'MONDO': 12638, 'ORPHANET': 234, 'DIS_B (blinded)': 149}


## Rare / common labels (frozen LLM-assisted annotation)
Methods: *13,021 diseases, 10,489 rare and 2,532 common (19.4%)*.

In [4]:
lab = pd.read_parquet(RAW / "llm_annotations/disease_labels.parquet")
assert set(lab.disease_id) == set(net.disease_id)
cls = lab.disease_class.value_counts()
print(cls.to_dict(), f"common share {cls['common'] / len(lab):.1%}")
print("rare label source:", lab.rare_source.value_counts().to_dict())
print("cancer label source:", lab.cancer_source.value_counts().to_dict(), "| is_cancer:", int(lab.is_cancer.sum()))
assert (cls["rare"], cls["common"]) == (10489, 2532)

{'rare': 10489, 'common': 2532} common share 19.4%
rare label source: {'rule': 12079, 'agent': 619, 'curated': 323}
cancer label source: {'rule': 12167, 'agent': 854} | is_cancer: 1213


## Drugs

In [5]:
dr = pd.read_parquet(RAW / "drugs/drugs.parquet")
moa = pd.read_parquet(RAW / "drugs/drug_moa_decision.parquet")
print(len(dr), "drugs;", dr.drug_id.str.startswith("CHEMBL:").sum(), "ChEMBL,", dr.drug_id.str.startswith("DRUG_B").sum(), "blinded")
print("MoA filter decision:", moa.decision.value_counts().to_dict())
assert set(t.drug_id) <= set(dr.drug_id)
print("drugs used in trial triples:", t.drug_id.nunique())

38389 drugs; 7806 ChEMBL, 30583 blinded
MoA filter decision: {'keep': 34865, 'exclude': 2535, 'flag_review': 989}
drugs used in trial triples: 38389


## Disease–gene and disease–symptom associations
Supplementary tables of input sources (tier-1, after capping). Source names are replaced by opaque `source_code`s; each row below equals one row of the corresponding Supplementary table.

In [6]:
g1 = pd.read_parquet(RAW / "associations/disease_gene_tier1_sources.parquet")
s1 = pd.read_parquet(RAW / "associations/disease_symptom_tier1_sources.parquet")
def by_source(x, item):
    n_src = x.groupby("disease_id").source_code.nunique()
    only = x[x.disease_id.isin(n_src[n_src == 1].index)].groupby("source_code").disease_id.nunique()
    out = x.groupby("source_code").agg(associations=(item, "size"), diseases=("disease_id", "nunique"), items=(item, "nunique"))
    out["exclusive_diseases"] = only
    return out.sort_values("associations", ascending=False)
print(by_source(g1, "gene_id"))
cg = g1[["disease_id", "gene_id"]].drop_duplicates()
print("combined:", len(cg), cg.disease_id.nunique(), cg.gene_id.nunique())
assert (len(cg), cg.disease_id.nunique(), cg.gene_id.nunique()) == (18287, 9426, 5656)

                associations  diseases  items  exclusive_diseases
source_code                                                      
SRC_F7D33DAC78          8150      4090   4421                1219
SRC_29BFF1D116          8009      5421   4063                 378
SRC_9F47AC4C6D          6903      6033   4157                 605
SRC_95D2C9F3F0          6019      5818   4328                 494
SRC_3910863E20          3413       783    995                 298
combined: 18287 9426 5656


In [7]:
print(by_source(s1, "hpo_id").drop(columns="exclusive_diseases"))
cs = s1[["disease_id", "hpo_id"]].drop_duplicates()
print("combined:", len(cs), cs.disease_id.nunique(), cs.hpo_id.nunique())
assert (len(cs), cs.disease_id.nunique(), cs.hpo_id.nunique()) == (167798, 8991, 10199)

                associations  diseases  items
source_code                                  
SRC_46BA6479D6        147952      7031   9622
SRC_D31701DD32        123388      8099   8800
combined: 167798 8991 10199


Final association tables (all evidence tiers; tier 9 = MONDO-propagation pass). Methods: *propagation adds 39,894 associations to 986 diseases*; SI: *10,994 of the 13,021 diseases (84.4%) carry a gene, 10,281 (79.0%) an HPO phenotype, 8,254 (63.4%) both, 4,767 (36.6%) miss one, none miss both*.

In [8]:
dg = pd.read_parquet(RAW / "associations/disease_gene.parquet")
ds = pd.read_parquet(RAW / "associations/disease_symptom.parquet")
print("gene tiers:", dg.evidence_tier.value_counts().sort_index().to_dict(), "| symptom tiers:", ds.evidence_tier.value_counts().sort_index().to_dict())
prop = ds[ds.evidence_tier == 9]
print("propagation:", len(prop), prop.disease_id.nunique())
N = set(net.disease_id); G = set(dg.disease_id) & N; H = set(ds.disease_id) & N
cov = {"gene": len(G), "hpo": len(H), "both": len(G & H), "missing_one": len(N) - len(G & H), "missing_both": len(N - G - H)}
print(cov, {k: f"{v / len(N):.1%}" for k, v in cov.items()})
assert (len(prop), prop.disease_id.nunique()) == (39894, 986)
assert cov == {"gene": 10994, "hpo": 10281, "both": 8254, "missing_one": 4767, "missing_both": 0}

gene tiers: {1: 22564, 2: 2152, 3: 1256} | symptom tiers: {1: 204500, 9: 39894}
propagation: 39894 986
{'gene': 10994, 'hpo': 10281, 'both': 8254, 'missing_one': 4767, 'missing_both': 0} {'gene': '84.4%', 'hpo': '79.0%', 'both': '63.4%', 'missing_one': '36.6%', 'missing_both': '0.0%'}


## Gene-level features
Methods: *HPA single-cell-type expression (17,294 genes × 154 cell types) and Pfam domains (18,506 genes, 8,843 families)*.

In [9]:
hpa = pd.read_parquet(RAW / "genes/gene_hpa.parquet")
pf = pd.read_parquet(RAW / "genes/gene_pfam.parquet")
go = pd.read_parquet(RAW / "genes/gene_go.parquet")
print("HPA", hpa.shape[0], "genes x", hpa.shape[1] - 1, "cell types")
print("Pfam", pf.gene_id.nunique(), "genes,", pf.pfam_id.nunique(), "families")
print("GO", go.gene_id.nunique(), "genes;", go.namespace.value_counts().to_dict())
assert (hpa.shape[0], hpa.shape[1] - 1) == (17294, 154)
assert (pf.gene_id.nunique(), pf.pfam_id.nunique()) == (18506, 8843)

HPA 17294 genes x 154 cell types
Pfam 18506 genes, 8843 families
GO 6169 genes; {'biological_process': 29689, 'molecular_function': 17528, 'cellular_component': 12095}


## Ontologies
Pinned OBO files are identified by MD5 (`MD5SUMS.txt`); `mondo.obo`, `mondo_propagation.obo` and `go-basic.obo` are in the Zenodo bundle and are checked only if present.

In [10]:
ont = RAW / "ontologies"
for line in (ont / "MD5SUMS.txt").read_text().splitlines():
    md5, name = line.split()[:2]
    p = REPO / "data/zenodo/00_raw/ontologies" / name.split(":")[-1] if name.startswith("zenodo:") else ont / name
    if not p.exists():
        print(f"{p.name:28s} not present (Zenodo)"); continue
    ok = hashlib.md5(p.read_bytes()).hexdigest() == md5
    n_terms = p.read_text().count("[Term]") if p.suffix == ".obo" else None
    print(f"{p.name:28s} md5 ok={ok} terms={n_terms}")
for f in sorted((ont / "ancestors").glob("*.parquet")):
    print(f.name, len(pd.read_parquet(f)))

hp.obo                       md5 ok=True terms=20413
go-basic.obo                 md5 ok=True terms=48329
mondo_propagation.obo        md5 ok=True terms=63278
mondo.obo                    md5 ok=True terms=60843
go_bp_semi_ancestors.parquet md5 ok=True terms=None
go_bp_semi_nodes.csv         md5 ok=True terms=None
go_bp_semi_term_remap.csv    md5 ok=True terms=None
go_cc_ancestors.parquet      md5 ok=True terms=None
go_cc_nodes.csv              md5 ok=True terms=None
go_cc_term_remap.csv         md5 ok=True terms=None
go_mf_ancestors.parquet      md5 ok=True terms=None
go_mf_nodes.csv              md5 ok=True terms=None
go_mf_term_remap.csv         md5 ok=True terms=None
hpo_ancestors.parquet        md5 ok=True terms=None
hpo_nodes.csv                md5 ok=True terms=None
hpo_term_remap.csv           md5 ok=True terms=None
go_bp_semi_ancestors.parquet 480463
go_cc_ancestors.parquet 46358
go_mf_ancestors.parquet 73818
hpo_ancestors.parquet 222576


## Frozen LLM labels for IxIDN-Matched curation
Methods: *the 4,302 candidate positives were judged; the 2,182 judged strong or plausible were kept*; re-judging 944 pairs gives the keep/drop agreement.

In [11]:
v = pd.read_parquet(RAW / "llm_annotations/ixidn_judge_verdicts.parquet")
print(len(v), v.verdict.value_counts().to_dict())
keep = lambda s: s.isin(["STRONG", "PLAUSIBLE"])
assert (keep(v.verdict) == v.keep).all()
rj = v[v.verdict_source == "expanded_pass_rescored"]
agree = int((keep(rj.verdict) == keep(rj.prior_verdict)).sum())
print("kept", int(v.keep.sum()), "| re-judged", len(rj), "agree", agree, f"= {agree / len(rj):.1%}")
assert (len(v), int(v.keep.sum()), len(rj), agree) == (4302, 2182, 944, 800)

4302 {'WEAK': 1354, 'STRONG': 1285, 'PLAUSIBLE': 897, 'SUSPECT': 709, 'NOT_IN_GRAPH': 57}
kept 2182 | re-judged 944 agree 800 = 84.7%
